In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
# Force working directory to the notebook's own folder
NOTEBOOK_DIR = Path().resolve()
os.chdir(NOTEBOOK_DIR)
DATA_DIR = NOTEBOOK_DIR / "data"

In [3]:
career_info = pd.read_csv("./data//Player Career Info.csv")
career_info.head()

,player,player_id,pos,ht_in_in,wt,birth_date,colleges,from,to,debut,hof
0,Hank Biasatti,biasaha01,G,71.0,175.0,1922-01-14,Assumption University,1947,1947,1946-11-01T00:00:00Z,False
1,Tommy Byrnes,byrneto01,F-G,75.0,175.0,1923-02-19,Seton Hall,1947,1951,1946-11-01T00:00:00Z,False
2,Bob Fitzgerald,fitzgbo01,F-C,77.0,190.0,1923-03-14,Seton Hall,1947,1949,1946-11-01T00:00:00Z,False
3,Dick Fitzgerald,fitzgdi01,F,74.0,175.0,1920-11-18,Fordham,1947,1948,1946-11-01T00:00:00Z,False
4,Frank Fucarino,fucarfr01,F,74.0,175.0,1920-07-24,Long Island University,1947,1947,1946-11-01T00:00:00Z,False


In [4]:
#calculate players career length
career_info_length = career_info[["player_id", "player", "pos", "ht_in_in", "wt", "from", "to", "hof"]].copy()
career_info_length["career_length"] = career_info_length["to"] - career_info_length["from"] + 1

career_info_length.head()

,player_id,player,pos,ht_in_in,wt,from,to,hof,career_length
0,biasaha01,Hank Biasatti,G,71.0,175.0,1947,1947,False,1
1,byrneto01,Tommy Byrnes,F-G,75.0,175.0,1947,1951,False,5
2,fitzgbo01,Bob Fitzgerald,F-C,77.0,190.0,1947,1949,False,3
3,fitzgdi01,Dick Fitzgerald,F,74.0,175.0,1947,1948,False,2
4,fucarfr01,Frank Fucarino,F,74.0,175.0,1947,1947,False,1


In [5]:
totals = pd.read_csv("./data/Player Totals.csv")
totals.head()

,season,lg,player,player_id,age,team,pos,g,gs,mp,...,orb,drb,trb,ast,stl,blk,tov,pf,pts,trp_dbl
0,2026,NBA,Precious Achiuwa,achiupr01,26.0,SAC,C,73,57.0,1745.0,...,177.0,315.0,492.0,101,65.0,50.0,65.0,123.0,736,0.0
1,2026,NBA,Steven Adams,adamsst01,32.0,HOU,C,32,11.0,730.0,...,145.0,131.0,276.0,48,22.0,20.0,35.0,55.0,187,0.0
2,2026,NBA,Bam Adebayo,adebaba01,28.0,MIA,C,73,73.0,2365.0,...,149.0,583.0,732.0,232,86.0,49.0,120.0,122.0,1468,0.0
3,2026,NBA,Ochai Agbaji,agbajoc01,25.0,2TM,SG,62,13.0,973.0,...,46.0,95.0,141.0,47,24.0,17.0,31.0,93.0,314,0.0
4,2026,NBA,Ochai Agbaji,agbajoc01,25.0,TOR,SG,42,13.0,650.0,...,33.0,62.0,95.0,30,17.0,11.0,19.0,75.0,181,0.0


In [6]:
stats_cols = ["g", "gs", "mp", "fg", "fga", "x3p", "x3pa", "x2p", "x2pa",
            "ft", "fta", "orb", "drb", "trb", "ast", "stl", "blk", "tov",
            "pf", "pts", "trp_dbl"]

# sum columns to find career totals then change the column name to align
totals_career = totals.groupby("player_id")[stats_cols].sum().add_prefix("career_").reset_index()

# rename player_id column because we just changed all columns to start with career_
totals_career = totals_career.rename(columns={"career_player_id": "player_id"})

totals_career.head()

,player_id,career_g,career_gs,career_mp,career_fg,career_fga,career_x3p,career_x3pa,career_x2p,career_x2pa,...,career_orb,career_drb,career_trb,career_ast,career_stl,career_blk,career_tov,career_pf,career_pts,career_trp_dbl
0,abdelal01,385,105.0,5017.0,983,1940,0.0,6.0,983.0,1934.0,...,446.0,851.0,1297.0,125,111.0,107.0,389.0,777.0,2299,0.0
1,abdulka01,1560,789.0,57446.0,15837,28307,1.0,18.0,6692.0,11689.0,...,2975.0,9394.0,17440.0,5660,1160.0,3189.0,2527.0,4657.0,38387,21.0
2,abdulma01,779,0.0,19965.0,3704,8418,0.0,0.0,0.0,0.0,...,18.0,39.0,2234.0,3684,26.0,6.0,0.0,2137.0,9348,1.0
3,abdulma02,586,336.0,15628.0,3514,7943,474.0,1339.0,3040.0,6604.0,...,219.0,868.0,1087.0,2079,487.0,46.0,963.0,1106.0,8553,0.0
4,abdulta01,321,213.0,6826.0,1049,2519,22.0,101.0,1027.0,2418.0,...,428.0,723.0,1151.0,388,263.0,121.0,442.0,688.0,2662,0.0


In [7]:
totals_career["career_fg_pct"] = totals_career["career_fg"] / totals_career["career_fga"]
totals_career["career_x3p_pct"] = totals_career["career_x3p"] / totals_career["career_x3pa"]
totals_career["career_ft_pct"] = totals_career["career_ft"] / totals_career["career_fta"]
totals_career["career_pts_per_g"] = totals_career["career_pts"] / totals_career["career_g"]
totals_career["career_ast_per_g"] = totals_career["career_ast"] / totals_career["career_g"]
totals_career["career_trb_per_g"] = totals_career["career_trb"] / totals_career["career_g"]

totals_career.head()

,player_id,career_g,career_gs,career_mp,career_fg,career_fga,career_x3p,career_x3pa,career_x2p,career_x2pa,...,career_tov,career_pf,career_pts,career_trp_dbl,career_fg_pct,career_x3p_pct,career_ft_pct,career_pts_per_g,career_ast_per_g,career_trb_per_g
0,abdelal01,385,105.0,5017.0,983,1940,0.0,6.0,983.0,1934.0,...,389.0,777.0,2299,0.0,0.506701,0.000000,0.705508,5.971429,0.324675,3.368831
1,abdulka01,1560,789.0,57446.0,15837,28307,1.0,18.0,6692.0,11689.0,...,2527.0,4657.0,38387,21.0,0.559473,0.055556,0.721410,24.607051,3.628205,11.179487
2,abdulma01,779,0.0,19965.0,3704,8418,0.0,0.0,0.0,0.0,...,0.0,2137.0,9348,1.0,0.440010,NaN,0.758109,12.000000,4.729140,2.867779
3,abdulma02,586,336.0,15628.0,3514,7943,474.0,1339.0,3040.0,6604.0,...,963.0,1106.0,8553,0.0,0.442402,0.353996,0.905254,14.595563,3.547782,1.854949
4,abdulta01,321,213.0,6826.0,1049,2519,22.0,101.0,1027.0,2418.0,...,442.0,688.0,2662,0.0,0.416435,0.217822,0.717881,8.292835,1.208723,3.585670


In [8]:
advanced = pd.read_csv("./data/Advanced.csv")
advanced.head()

,season,lg,player,player_id,age,team,pos,g,gs,mp,...,tov_percent,usg_percent,ows,dws,ws,ws_48,obpm,dbpm,bpm,vorp
0,2026,NBA,Precious Achiuwa,achiupr01,26.0,SAC,C,73,57.0,1745.0,...,9.0,17.6,1.8,1.2,2.9,0.081,-0.5,-1.1,-1.6,0.2
1,2026,NBA,Steven Adams,adamsst01,32.0,HOU,C,32,11.0,730.0,...,16.7,12.1,1.1,1.0,2.1,0.141,-0.1,-0.3,-0.5,0.3
2,2026,NBA,Bam Adebayo,adebaba01,28.0,MIA,C,73,73.0,2365.0,...,8.3,25.0,2.8,3.6,6.4,0.129,1.5,0.4,2.0,2.4
3,2026,NBA,Ochai Agbaji,agbajoc01,25.0,2TM,SG,62,13.0,973.0,...,9.4,14.6,0.1,0.8,0.9,0.046,-3.5,-0.5,-4.0,-0.5
4,2026,NBA,Ochai Agbaji,agbajoc01,25.0,TOR,SG,42,13.0,650.0,...,9.4,13.3,0.0,0.7,0.6,0.048,-4.8,0.1,-4.7,-0.5


In [9]:
# advanced_efficiency_cols cant be added together like the advanced_summation_cols can
advanced_summation_cols = ["ows", "dws", "ws", "vorp"]
advanced_efficiency_cols = ["per", "ts_percent", "bpm", "obpm", "dbpm", "usg_percent"]

# career sums for the stats that are cumulative/ the ones we can just add together
advanced_sums = advanced.groupby("player_id")[advanced_summation_cols].sum().add_prefix("career_").reset_index()
advanced_sums = advanced_sums.rename(columns={"career_player_id": "player_id"})

# multiply the rate/efficiency percentage columns by minutes
advanced_w = advanced.dropna(subset=["mp"]).copy()
weighted_stats = advanced_w[advanced_efficiency_cols].mul(advanced_w["mp"], axis=0)

# add player_id and minutes played back to the dataset and then find the sum
weighted_stats[["player_id", "mp"]] = advanced_w[["player_id", "mp"]]
grouped = weighted_stats.groupby("player_id").sum()

# divide the sum of the weighted stats that we calculated by the total minutes
advanced_weighted = (grouped[advanced_efficiency_cols].div(grouped["mp"], axis=0).add_prefix("career_avg_").reset_index())

# identify players' best season through win share/player impact
peak = advanced.groupby("player_id")["ws"].max().reset_index().rename(columns={"ws": "peak_season_ws"})

advanced_sums.head()

,player_id,career_ows,career_dws,career_ws,career_vorp
0,abdelal01,1.0,6.2,7.1,-2.4
1,abdulka01,179.0,94.5,273.3,85.9
2,abdulma01,18.4,12.9,31.2,0.0
3,abdulma02,16.9,8.4,25.1,4.6
4,abdulta01,-0.6,6.2,5.5,-1.1


In [10]:
# only need number of seasons because we can join the data later to then compare with other columns
# lots of repetitive columns in this set since if a player played 10 seasons they have 10 rows and we already have career stats
season_info = pd.read_csv("./data/Player Season Info.csv")
seasons_played = season_info.groupby("player_id")["season"].nunique().reset_index()
seasons_played = seasons_played.rename(columns={"season": "n_seasons"})

seasons_played.head()

,player_id,n_seasons
0,abdelal01,5
1,abdulka01,20
2,abdulma01,10
3,abdulma02,9
4,abdulta01,6


In [11]:
# checking all star status and counting number of times
all_star = pd.read_csv("./data/All-Star Selections.csv")
all_star_count = all_star.groupby("player_id").size().reset_index(name="n_all_star")

end_of_season = pd.read_csv("./data/End of Season Teams.csv")

# sum awards
all_nba = end_of_season[end_of_season["type"] == "All-NBA"].groupby("player_id").size().reset_index(name="n_all_nba")
all_def = end_of_season[end_of_season["type"] == "All-Defense"].groupby("player_id").size().reset_index(name="n_all_defense")
all_rookie = end_of_season[end_of_season["type"] == "All-Rookie"].groupby("player_id").size().reset_index(name="n_all_rookie")

# finds All-NBA AND 1st team 
first_team_all_nba = end_of_season[(end_of_season["type"] == "All-NBA") & (end_of_season["number_tm"] == "1st")]
first_team_count = first_team_all_nba.groupby("player_id").size().reset_index(name="n_first_team_all_nba")

# sum awards and award share
awards = pd.read_csv("./data/Player Award Shares.csv")
award_sum = awards.groupby("player_id").agg(max_award_share=("share", "max"), total_award_share=("share", "sum"), n_award_wins=("winner", "sum"),).reset_index()

# keep all awards with MVP in the name, not case sensitive
# ~awards flips the entire column of true/false values similar to NOT but for an entire column of values
mvp = awards[awards["award"].str.contains("mvp", case = False, na = False) & ~awards["award"].str.contains("clutch|def|roy|6th|sixth", case=False, na=False)]

mvp_sum = mvp.groupby("player_id").agg(mvp_max_share=("share", "max"),n_mvp_wins=("winner", "sum"),).reset_index()

all_star_count.head()

,player_id,n_all_star
0,abdulka01,19
1,abdulma01,1
2,abdursh01,1
3,adamsal01,1
4,adamsmi01,1


In [12]:
# Now, we can finally combine all the data into one dataset

NBA_data = career_info_length.merge(totals_career, on="player_id", how="left")
NBA_data = NBA_data.merge(advanced_sums, on="player_id", how="left")
NBA_data = NBA_data.merge(advanced_weighted, on="player_id", how="left")
NBA_data = NBA_data.merge(peak, on="player_id", how="left")
NBA_data = NBA_data.merge(seasons_played, on="player_id", how="left")
NBA_data = NBA_data.merge(all_star_count, on="player_id", how="left")
NBA_data = NBA_data.merge(all_nba, on="player_id", how="left")
NBA_data = NBA_data.merge(all_def, on="player_id", how="left")
NBA_data = NBA_data.merge(all_rookie, on="player_id", how="left")
NBA_data = NBA_data.merge(first_team_count, on="player_id", how="left")
NBA_data = NBA_data.merge(award_sum, on="player_id", how="left")
NBA_data = NBA_data.merge(mvp_sum, on="player_id", how="left")

NBA_data.head()

,player_id,player,pos,ht_in_in,wt,from,to,hof,career_length,career_g,...,n_all_star,n_all_nba,n_all_defense,n_all_rookie,n_first_team_all_nba,max_award_share,total_award_share,n_award_wins,mvp_max_share,n_mvp_wins
0,biasaha01,Hank Biasatti,G,71.0,175.0,1947,1947,False,1,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,byrneto01,Tommy Byrnes,F-G,75.0,175.0,1947,1951,False,5,368,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,fitzgbo01,Bob Fitzgerald,F-C,77.0,190.0,1947,1949,False,3,138,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,fitzgdi01,Dick Fitzgerald,F,74.0,175.0,1947,1948,False,2,61,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,fucarfr01,Frank Fucarino,F,74.0,175.0,1947,1947,False,1,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# for players that didnt win an award or appear in All-Star selection, we want to just put in 0 for the missing value
missing_cols = ["n_all_star", "n_all_nba", "n_all_defense", "n_all_rookie", "n_first_team_all_nba", "n_award_wins", "n_mvp_wins", "max_award_share", "total_award_share", "mvp_max_share"]
NBA_data[missing_cols] = NBA_data[missing_cols].fillna(0)

# drop any player with no career stats
NBA_data = NBA_data.dropna(subset = ["career_g"])

NBA_data.shape

(5416, 58)

In [14]:
print(list(NBA_data.columns))

['player_id', 'player', 'pos', 'ht_in_in', 'wt', 'from', 'to', 'hof', 'career_length', 'career_g', 'career_gs', 'career_mp', 'career_fg', 'career_fga', 'career_x3p', 'career_x3pa', 'career_x2p', 'career_x2pa', 'career_ft', 'career_fta', 'career_orb', 'career_drb', 'career_trb', 'career_ast', 'career_stl', 'career_blk', 'career_tov', 'career_pf', 'career_pts', 'career_trp_dbl', 'career_fg_pct', 'career_x3p_pct', 'career_ft_pct', 'career_pts_per_g', 'career_ast_per_g', 'career_trb_per_g', 'career_ows', 'career_dws', 'career_ws', 'career_vorp', 'career_avg_per', 'career_avg_ts_percent', 'career_avg_bpm', 'career_avg_obpm', 'career_avg_dbpm', 'career_avg_usg_percent', 'peak_season_ws', 'n_seasons', 'n_all_star', 'n_all_nba', 'n_all_defense', 'n_all_rookie', 'n_first_team_all_nba', 'max_award_share', 'total_award_share', 'n_award_wins', 'mvp_max_share', 'n_mvp_wins']


In [15]:
hof_players = NBA_data[NBA_data["hof"] == True]
hof_players

,player_id,player,pos,ht_in_in,wt,from,to,hof,career_length,career_g,...,n_all_star,n_all_nba,n_all_defense,n_all_rookie,n_first_team_all_nba,max_award_share,total_award_share,n_award_wins,mvp_max_share,n_mvp_wins
111,fulksjo01,Joe Fulks,F-C,77.0,190.0,1947,1954,True,8,489,...,2.0,1.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
165,jeannbu01,Buddy Jeannette,G,71.0,175.0,1948,1950,True,3,139,...,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
170,braunca01,Carl Braun,G-F,77.0,180.0,1948,1962,True,15,788,...,5.0,1.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
182,phillan01,Andy Phillip,G-F,74.0,195.0,1948,1958,True,11,771,...,5.0,2.0,0.0,0.0,0.0,0.000,0.000,0.0,0.000,0.0
239,mikange01,George Mikan,C,82.0,245.0,1949,1956,True,8,439,...,4.0,5.0,0.0,0.0,5.0,0.000,0.000,0.0,0.000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3386,mingya01,Yao Ming,C,90.0,310.0,2003,2011,True,9,486,...,8.0,5.0,0.0,1.0,0.0,0.692,0.730,0.0,0.001,0.0
3433,wadedw01,Dwyane Wade,G,76.0,220.0,2004,2019,True,16,1121,...,13.0,8.0,3.0,1.0,2.0,0.562,1.425,0.0,0.562,0.0
3435,anthoca01,Carmelo Anthony,F,79.0,238.0,2004,2022,True,19,1337,...,10.0,6.0,0.0,1.0,0.0,0.729,1.236,0.0,0.393,0.0
3439,boshch01,Chris Bosh,F-C,83.0,235.0,2004,2016,True,13,893,...,11.0,1.0,0.0,1.0,0.0,0.068,0.139,0.0,0.033,0.0


In [16]:
NBA_data["career_ts_percent"] = NBA_data["career_pts"] / (2 * (NBA_data["career_fga"] + 0.44 * NBA_data["career_fta"]))

In [17]:
NBA_data["hof_numeric"] = NBA_data["hof"].astype(int)

In [18]:
#calculate possible input numeric columns

numeric_cols = NBA_data.select_dtypes(include = [np.number]).columns.tolist()
numeric_cols.remove("hof_numeric")

correlations = NBA_data[numeric_cols].corrwith(NBA_data["hof_numeric"]).sort_values(ascending=False)

correlations

n_all_star                0.674930
n_all_nba                 0.567741
career_ws                 0.528292
career_fta                0.527318
career_ft                 0.523956
career_ows                0.519547
mvp_max_share             0.493037
career_pts                0.473789
career_dws                0.472871
career_fg                 0.472285
career_fga                0.467847
peak_season_ws            0.449264
n_first_team_all_nba      0.447993
career_trb                0.429193
career_mp                 0.407602
career_pts_per_g          0.393517
career_vorp               0.392774
career_ast                0.391393
total_award_share         0.387387
career_pf                 0.382615
n_award_wins              0.375290
max_award_share           0.365302
n_all_defense             0.328783
career_trb_per_g          0.325110
career_g                  0.322152
n_seasons                 0.311404
n_mvp_wins                0.309162
career_length             0.303263
career_ast_per_g    

In [19]:
# combined list of selected features of the correlations we calculated and cedric's selected

selected_features = [
    "career_vorp",
    "n_seasons",
    "career_pf",
    "career_fta",
    "career_x3pa",
    "mvp_max_share",
    "career_dws",
    "career_length",
    "career_ast_per_g",
    "n_all_nba",
    "n_all_star",
    "peak_season_ws",
    "career_trb",
    "career_trb_per_g",
    "career_g"
]

eligible = NBA_data["to"] <= 2018

X = NBA_data.loc[eligible, selected_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = NBA_data.loc[eligible, "hof_numeric"]

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 542, stratify = y)

In [21]:
# to handle the class imbalance of hof or not we need to use sample weights

sample_weights = compute_sample_weight(class_weight = "balanced", y = y_train)

In [22]:
# create the Gradient Boost model 

gb_model = GradientBoostingClassifier(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 3,
    random_state = 542
)

# fit the model

gb_model.fit(X_train, y_train, sample_weight = sample_weights)

GradientBoostingClassifier(learning_rate=0.05, n_estimators=200,
                           random_state=542)

In [23]:
# create predictions

y_pred = gb_model.predict(X_test)
y_proba = gb_model.predict_proba(X_test)[:, 1]

In [24]:
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       794
           1       0.63      0.79      0.70        34

    accuracy                           0.97       828
   macro avg       0.81      0.89      0.84       828
weighted avg       0.98      0.97      0.97       828

ROC-AUC: 0.9731441695065937
Confusion matrix:
[[778  16]
 [  7  27]]


In [25]:
# fill missing values

X_full = NBA_data[selected_features].replace([np.inf, -np.inf], np.nan).fillna(0)

NBA_data["hof_probability"] = gb_model.predict_proba(X_full)[:, 1]

# calculate future candidates with percentage

future_candidates = NBA_data[NBA_data["hof_numeric"] == 0].sort_values("hof_probability", ascending=False)
future_candidates[["player", "to", "hof_probability"]].head(20)

,player,to,hof_probability
3927,Paul George,2026,0.994704
3871,James Harden,2026,0.994704
3569,Chris Paul,2026,0.994704
3864,Stephen Curry,2026,0.994704
3445,LeBron James,2026,0.994656
4144,Giannis Antetokounmpo,2026,0.994656
4313,Nikola Jokić,2026,0.994656
4068,Anthony Davis,2026,0.994179
3670,Kyle Lowry,2026,0.991923
3811,Kevin Love,2026,0.991420


In [26]:
# we can clearly see that the probability percentages for some are WAY too high and some have most likely missed their chances (retired long ago)
# we need to adjust the model

eligible_df = NBA_data[NBA_data['to'] <= 2018].copy()
active_df = NBA_data[NBA_data['to'] > 2018].copy()

In [27]:
X = eligible_df[selected_features]
y = eligible_df['hof']

In [28]:
# tests of 243 combos since 3x3x3x3x3 = 243

paramGrid = {
    "max_iter": [200, 300, 400],
    "max_depth": [5, 7, 10],
    "learning_rate": [0.01, 0.03, 0.05],
    "min_samples_leaf": [5, 10, 20],
    "l2_regularization": [0.0, 0.1, 1.0],
}

# split 5 times but randomize first in case of any grouping of years prior to ensure randomization

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 542)

# sample weights

sampleWeights = compute_sample_weight(class_weight = "balanced", y = y)

# create the hist gradient boost classifier

histSearch = HistGradientBoostingClassifier(random_state = 542)

# run grid search

gridSearch = GridSearchCV(
    estimator = histSearch,
    param_grid = paramGrid,
    scoring = "roc_auc",
    cv = cv,
    n_jobs = -1,
)

# fit the grid search using the sample weights

gridSearch.fit(X, y, sample_weight = sampleWeights)

print("Best ROC-AUC:", gridSearch.best_score_)
print("Best params:", gridSearch.best_params_)

c:\Users\bordc\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\bordc\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\bordc\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bordc\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Best ROC-AUC: 0.9692070022236725
Best params: {'l2_regularization': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'max_iter': 300, 'min_samples_leaf': 10}


In [29]:
# retrain test set

X_train_calib, X_calib, y_train_calib, y_calib = train_test_split(
    X, y, test_size = 0.3, random_state = 542, stratify = y
)

sampleWeightsTrain = compute_sample_weight(class_weight = "balanced", y = y_train_calib)

In [30]:
# find best grid params

best = gridSearch.best_params_

# tuned hist gradient boosting classifier

tuned_hist_gb = HistGradientBoostingClassifier(
    random_state = 542,
    l2_regularization = best["l2_regularization"],
    learning_rate = best["learning_rate"],
    max_depth = best["max_depth"],
    max_iter = best["max_iter"],
    min_samples_leaf = best["min_samples_leaf"]
)

# fitting the model

tuned_hist_gb.fit(X_train_calib, y_train_calib, sample_weight = sampleWeightsTrain)

# calibrate using the tuned model

calibratedGb = CalibratedClassifierCV(tuned_hist_gb, method = "sigmoid")
calibratedGb.fit(X_calib, y_calib)

# make predictions after eligibility

eligibleActive = active_df[(active_df['career_g'] >= 400) | (active_df['career_ws'] >= 40.0)].copy()
eligibleActive["hof_prob"] = calibratedGb.predict_proba(eligibleActive[selected_features])[:, 1]

top_hof_candidates = eligibleActive.sort_values(by = "hof_prob", ascending = False)[["player", "hof_prob", "career_pts_per_g", "career_ws"]].copy()
top_hof_candidates["hof_prob"] = top_hof_candidates["hof_prob"].map("{:.1%}".format)
top_hof_candidates.head(20)

,player,hof_prob,career_pts_per_g,career_ws
3864,Stephen Curry,89.4%,24.815716,147.4
3433,Dwyane Wade,89.4%,21.347012,121.9
3734,Kevin Durant,89.3%,27.213942,193.0
3927,Paul George,89.3%,20.463492,92.1
3445,LeBron James,89.1%,26.781751,276.9
4144,Giannis Antetokounmpo,89.1%,24.056983,125.7
3569,Chris Paul,88.9%,16.830657,215.3
3871,James Harden,88.9%,23.932857,204.7
3820,Russell Westbrook,88.8%,20.622271,114.5
3435,Carmelo Anthony,88.5%,22.632012,116.5


In [31]:
calib_pred = tuned_hist_gb.predict(X_calib)
calib_proba = tuned_hist_gb.predict_proba(X_calib)[:, 1]

print(classification_report(y_calib, calib_pred))

print("Accuracy: ", accuracy_score(y_calib, calib_pred))
print("ROC-AUC: ", roc_auc_score(y_calib, calib_proba))

print(confusion_matrix(y_calib, calib_pred))

              precision    recall  f1-score   support

       False       0.99      0.97      0.98      1191
        True       0.55      0.80      0.66        51

    accuracy                           0.97      1242
   macro avg       0.77      0.89      0.82      1242
weighted avg       0.97      0.97      0.97      1242

Accuracy:  0.965378421900161
ROC-AUC:  0.9519599611465073
[[1158   33]
 [  10   41]]
